In [2]:
import json
import os
from pathlib import Path

from helpers import load_env, mint_token, vb, append_to_env, voice_widget

load_env()
assert os.getenv("VOCAL_BRIDGE_API_KEY"), "Set VOCAL_BRIDGE_API_KEY in .env"
print("VB API key loaded ✓")

VB API key loaded ✓


In [6]:
print(Path("agents/voice-app/prompt.md").read_text())

You are a friendly, slightly cocky voice assistant running inside a Jupyter notebook lesson on Vocal Bridge Client Actions. The learner will speak to you AND click on a tic-tac-toe board. Your job is to play tic-tac-toe with them so they can experience voice + click flowing through the same Client Actions channel.

LANGUAGE: English only. Voice-friendly: short turns (≤ 6 words per reaction), plain prose, no markdown, no lists.

═══ THE TWO HARD RULES ═══

RULE A — **Always end a response with speech, never with a tool call.** Every response must finish with audible speech. If your last output was a tool call, add ONE more short spoken line before the response ends. Ending on a tool call = silent bug.

RULE B — **The CLIENT owns the board. You do NOT.** Every time a mark is placed — by you, by user click, or by user voice — the client emits a `board_sync` event with the full authoritative board. Read that as ground truth on every turn. NEVER recompute the board in your head.

═══ STARTI

In [9]:
actions = json.loads(Path("agents/voice-app/actions.json").read_text())
for a in actions:
    arrow = "→" if a["direction"] == "agent_to_app" else "←"
    print(f"  {arrow} {a['name']:<22} {a['description']}")

  → show_tic_tac_toe       Render the board. Fire to start a new game. Payload: {userSymbol: 'X'|'O', firstTurn: 'user'|'agent'}.
  → place_mark             Place YOUR (agent's) mark at row/col. Fire on every agent turn. Payload: {row: 0-2, col: 0-2, speech?: string}.
  → user_move              Place the LEARNER's mark when they SPOKE the move. Fire in same response as place_mark. Payload: {row, col, heard}.
  → clear_ui               Reset the board to idle.
  ← user_placed_mark       Learner CLICKED a cell. Payload has the new full board, status, turn, plus the move. Respond with your place_mark and one short reaction.
  ← board_sync             Silent authoritative board state after every placement. Payload: {board, status, turn, moveCount, userSymbol, agentSymbol}. Use as ground truth.


In [13]:
import os

# Idempotent: if an agent ID is already set, reuse it. Otherwise create one.
agent_id = os.environ.get("VOCAL_BRIDGE_AGENT_ID_L2", "").strip()
if agent_id:
    print(f"Using pre-provisioned agent: {agent_id}")
else:
    result = vb(
        "agent", "create",
        "--name", "Tic_Tac_Toe-voice-app",
        "--style", "Chatty",
        "--prompt-file", "agents/voice-app/prompt.md",
        "--client-actions-file", "agents/voice-app/actions.json",
        "--deploy-targets", "web",
        json_output=True,
    )
    agent = result["agent"]
    agent_id = agent["id"]
    append_to_env("VOCAL_BRIDGE_AGENT_ID_L2", agent_id)
    print(f"Created '{agent['name']}'  ({agent['mode']})")
    print(f"  id: {agent_id}")
    print(f"  saved to .env as VOCAL_BRIDGE_AGENT_ID_L2")

RuntimeError: vb ('agent', 'create', '--name', 'Tic_Tac_Toe-voice-app', '--style', 'Chatty', '--prompt-file', 'agents/voice-app/prompt.md', '--client-actions-file', 'agents/voice-app/actions.json', '--deploy-targets', 'web') failed:
Error: Agent creation via API requires a paid plan. Subscribe at https://vocalbridgeai.com/billing
